# 1. Quick Tour — What Can pytanga Do?

A rapid, example-driven tour through pytanga's major capabilities to give
new users an immediate sense of what the library can do. Glance at each major area —
not to master it, but to see the big picture and know where to dive deeper. Each
section ends with a reference to the tutorial that covers the topic in full.

## Setup

Make sure pytanga is installed. See [Tutorial 2](../02_installation/02_installation.ipynb)
for detailed setup instructions.

In [ ]:
import pytanga
from pytanga.basis import BasisE3, BasisPGA3
import math

---

## 1. Algebra & Multivectors

Create an algebra, build multivectors from strings, compute products, extract grades.

📖 **Full tutorial:** [Tutorial 3 — Algebra and Multivectors](../03_algebra_core/03_algebra_core.ipynb)

In [ ]:
# Create a 3D Euclidean algebra using a basis class
alg = BasisE3()
e1, e2, e3 = alg.e1, alg.e2, alg.e3

print(f"Algebra: dim={alg.dim}, signature={alg.sig}, total blades={alg.algebra_dim}")

In [ ]:
# Build multivectors from strings
a = alg("e1 + 2 e2")
b = alg("e2 + e3")
print(f"a = {a}")
print(f"b = {b}")

In [ ]:
# Geometric, outer, and inner products
print(f"a * b  (geometric) = {a * b}")
print(f"a ^ b  (outer)      = {a ^ b}")
print(f"a | b  (inner)      = {a | b}")

In [ ]:
# Grade extraction and reverse
mv = a * b              # scalar + bivector
print(f"Full mv:    {mv}")
print(f"Grade 0:    {mv.grade(0)}")
print(f"Grade 2:    {mv.grade(2)}")
print(f"Reverse:    {~mv}")
print(f"Pruned:     {mv.prune()}")  # remove near-zero coefficients

---

## 2. Basis Classes

Use `BasisE3` for named blades and factory methods; glimpse `BasisPGA3` for plane-based GA.

📖 **Full tutorial:** [Tutorial 4 — The Four Basis Classes](../04_basis_classes/04_basis_classes.ipynb)

In [ ]:
# BasisE3 is already imported above — reuse it
E3 = BasisE3()

# Named blade attributes
print(f"e1       = {E3.e1}")
print(f"e12      = {E3.e12}")
print(f"e123     = {E3.e123}")     # pseudoscalar
print(f"e12 * e3 = {E3.e12 * E3.e3}")  # geometric product → pseudoscalar

In [ ]:
# Factory methods
v = E3.vector(1, 2, 3)
s = E3.scalar(5)
print(f"Vector: {v}")
print(f"Scalar: {s}")

In [ ]:
# Glimpse BasisPGA3 — plane-based GA with geometric factories
PGA = BasisPGA3()
point  = PGA.point(1, 2, 3)      # point at (1, 2, 3)
plane  = PGA.plane(0, 0, 1, 5)    # z=5 plane
line   = PGA.line(0, 0, 0, 1, 0, 0)  # line through origin along x
print(f"PGA point: {point}")
print(f"PGA plane: {plane}")
print(f"PGA line:  {line}")

---

## 3. Rotors & Motions

Build a rotor, rotate a vector with the sandwich product.

📖 **Full tutorial:** [Tutorial 5 — Euclidean 3D](../05_euclidean_e3/05_euclidean_e3.ipynb)

In [ ]:
# Build a rotor: 90° rotation about the e3 axis
angle = math.pi / 2
rotor = E3.rotor(angle, E3.e3)
print(f"Rotor: {rotor}")

In [ ]:
# Apply via versor/sandwich product: R * v * ~R
v = E3.e1
rotated = rotor * v * ~rotor
print(f"Original: {v}")
print(f"Rotated:  {rotated}")   # should be approximately e2

In [ ]:
# Compose two rotations
rotor2 = E3.rotor(angle, E3.e1)
combined = rotor2 * rotor
rotated2 = combined * v * ~combined
print(f"Two rotations: {rotated2}")

---

## 4. Geometry Submodule

Create a `Point`, `Line`, and `Sphere` with `pytanga.geometry.create()`; round-trip with `analyze()`.

📖 **Full tutorial:** [Tutorial 15 — Geometry Submodule](../15_geometry/15_geometry.ipynb)

In [ ]:
from pytanga.geometry import Point, Direction, Line, Plane, Sphere, Rotor, create, analyze

# Create geometric entities (algebra-independent data model)
p = Point(1, 2, 3)
d = Direction(0, 0, 1)
l = Line(point=p, direction=d)
s = Sphere(center=Point(0, 0, 0), radius=2.5)

print(f"Point:  {p}")
print(f"Line:   {l}")
print(f"Sphere: {s}")

In [ ]:
# Convert entities to multivectors using the PGA3 basis class
pga = BasisPGA3()

point_mv = create(pga, p)
plane_mv = create(pga, Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)))

print(f"Point MV: {point_mv}")
print(f"Plane MV: {plane_mv}")

In [ ]:
# Round-trip: MV → analyze → Entity (must match original)
result = analyze(point_mv)
print(f"analyze(create(PGA, p)) = {result}")
print(f"Round-trip matches: {result == p}")

In [ ]:
# Analyze a rotor
r = Rotor(angle=math.pi / 4, axis=Direction(0, 0, 1))
print(f"Rotor entity: {r}")
print(f"As MV:        {create(pga, r)}")

---

## 5. Equation Solving

Solve `A * X = B` for an unknown multivector with `solve()`.

📖 **Full tutorial:** [Tutorial 12 — Equation Solving](../12_equation_solving/12_equation_solving.ipynb)

In [ ]:
from pytanga.solver.solve import solve

# Find the multiplicative inverse: A * X = 1
A = alg.random_mv(rng=42)
X = solve(A, 1.0, algebra=alg)

print(f"A = {A}")
print(f"X (inverse) = {X}")

# Verify: A * X should be ~1.0
check = A * X
check.prune()
print(f"A * X = {check}")

In [ ]:
# Solve A * X = B for a known right-hand side
B = alg.vector(0, 5, 0)
X2 = solve(A, B, algebra=alg)
check2 = A * X2
check2.prune()
print(f"A * X = {check2}")
print(f"Matches B: {check2.grade(1) == B}")

---

## 6. 3D Visualization

Fire up `pytanga.viz.Visualizer`, add entities, and export a standalone HTML figure.

📖 **Full tutorial:** [Tutorial 16 — 3D Visualization](../16_viz_scenes/16_viz_scenes.ipynb)

> **Note:** The visualizer opens an interactive Three.js viewer in your browser.
> If running headless or in a CI environment, skip the `viz.run()` cell and
> use the `SceneExporter` cell instead to produce a standalone HTML file.

In [ ]:
from pytanga.viz import Visualizer
from pytanga.geometry import Point, Sphere, Plane, Direction

# Create a visualizer
viz = Visualizer()

# Add a sphere, a plane, and a rotated vector (as raw MV)
viz.add(Sphere(center=Point(0, 0, 0), radius=2), color="#4488ff", opacity=0.3)
viz.add(Plane(point=Point(0, 0, 2), normal=Direction(0, 0, 1)), color="#44ff44", opacity=0.3)

# Build a rotated vector and add it (reusing the E3 algebra from earlier)
v_e1 = E3.e1 * 2               # vector (2, 0, 0)
rotor_viz = E3.rotor(math.pi / 4, E3.e3)
v_rotated = rotor_viz * v_e1 * ~rotor_viz   # vector (~1.41, 1.41, 0)

viz.add(E3.e1 * 2, color="#ff4444", label="original")
viz.add(v_rotated, color="#ff8844", label="rotated 45°")

# Open interactive browser window (blocks until Ctrl+C)
# viz.run()

In [ ]:
# Export as a standalone HTML file (works even without a browser)
from pytanga.viz.export import SceneExporter

exporter = SceneExporter(viz)
exporter.to_html("exports/quick_tour_demo.html")
print("Exported to exports/quick_tour_demo.html — open in any browser!")

---

## Where to Go Next

This quick tour scratched the surface. Each section links to a dedicated tutorial
that goes much deeper:

| Section | Full Tutorial |
|---------|---------------|
| Algebra & Multivectors | [Tutorial 3](../03_algebra_core/03_algebra_core.ipynb) |
| Basis Classes | [Tutorial 4](../04_basis_classes/04_basis_classes.ipynb) |
| Rotors & Motions | [Tutorial 5](../05_euclidean_e3/05_euclidean_e3.ipynb) |
| Geometry Submodule | [Tutorial 15](../15_geometry/15_geometry.ipynb) |
| Equation Solving | [Tutorial 12](../12_equation_solving/12_equation_solving.ipynb) |
| 3D Visualization | [Tutorial 16](../16_viz_scenes/16_viz_scenes.ipynb) |

Or start from the beginning with [Tutorial 2 — Installation](../02_installation/02_installation.ipynb).